# Speech Emotion Recognition — Train on Colab (or any GPU notebook platform)

Reproduces this repo's full pipeline end-to-end: SVM baseline -> LSTM-only -> CNN-only -> CNN-LSTM
hybrid, on RAVDESS + TESS. This notebook is a thin runner around the actual project code in `src/`
and `scripts/` -- it does not reimplement any logic, so results here match the repo's own numbers
(SVM 82.4%, LSTM-only 87.3%, CNN-only 84.4%, hybrid 89.9% test accuracy) exactly, and it stays in
sync automatically as `src/` changes.

**Why run this on Colab at all, if it already works locally?** Free GPU. The project's local
development machine has no `tensorflow-metal` support, so training there is CPU-only (the hybrid:
~15-20 min). On a Colab T4 GPU, the same training should take a couple of minutes.

The clone cell below already points at this project's repo
(`github.com/sonali9569/Speech-Emotion-Recognition`) — this notebook needs the repo's `src/` and
`scripts/` alongside it, the same way it needs them locally.


## 0. Check for a GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', bool(gpus), '-', gpus if gpus else '(none — Colab: Runtime > Change runtime type > GPU)')


## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/sonali9569/Speech-Emotion-Recognition.git project
%cd project
!pip install -q -r requirements.txt


## 2. Get the dataset onto this runtime

Two options — run WHICHEVER ONE CELL matches how you're providing the data this session, not both.

**Option A — direct upload.** Zip `data/raw/ravdess/` and `data/raw/tess/` yourself (or the raw
RAVDESS/TESS download zips), upload here. Fine for one-off runs; re-uploads every session.

**Option B — Google Drive.** Extract RAVDESS/TESS into your Drive once
(`MyDrive/ravdess/`, `MyDrive/tess/`), mount Drive, copy in. No re-upload needed on future runs —
just re-mount.


In [ ]:
# --- Option A: direct upload ---
from google.colab import files
import zipfile, os

print("Upload your RAVDESS zip (containing Actor_01..Actor_24 folders):")
uploaded = files.upload()
ravdess_zip = list(uploaded.keys())[0]
with zipfile.ZipFile(ravdess_zip) as z:
    z.extractall('data/raw/ravdess')

print("Upload your TESS zip (containing OAF_*/YAF_* folders):")
uploaded = files.upload()
tess_zip = list(uploaded.keys())[0]
with zipfile.ZipFile(tess_zip) as z:
    z.extractall('data/raw/tess')


In [ ]:
# --- Option B: Google Drive ---
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copytree('/content/drive/MyDrive/ravdess', 'data/raw/ravdess', dirs_exist_ok=True)
shutil.copytree('/content/drive/MyDrive/tess', 'data/raw/tess', dirs_exist_ok=True)


**Duplicate-folder check.** Both official RAVDESS/TESS zip downloads are known to extract with a
duplicate nested copy of every file — silently doubling every class's count if not caught. This
cell detects (not blindly assumes) whether that's happened here, by comparing top-level
actor/speaker folder counts against the full recursive file count.

In [ ]:
import os

def count_wavs(root):
    return sum(len(files) for _, _, files in os.walk(root) if any(f.endswith('.wav') for f in files))

ravdess_files = count_wavs('data/raw/ravdess')
tess_files = count_wavs('data/raw/tess')
print(f'RAVDESS: {ravdess_files} files (expected 1440)')
print(f'TESS: {tess_files} files (expected 2800)')

if ravdess_files not in (1440,) or tess_files not in (2800,):
    print('\n⚠️  Count mismatch — check for the duplicate-nested-folder issue: list subdirectories')
    print('    with `!find data/raw/ravdess -maxdepth 1` / `!find data/raw/tess -maxdepth 1` and')
    print('    remove any duplicate nested copy before continuing.')
else:
    print('\n✅ Counts match expected — no duplicate-folder issue here.')


## 3. Parse metadata, preprocess, extract features

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.parse import build_dataset_dataframe, add_audio_metadata
from src.data.split import stratified_split
from src.features.extract import batch_extract

df = build_dataset_dataframe('data/raw/ravdess', 'data/raw/tess')
df = add_audio_metadata(df)
df['split'] = stratified_split(df)
print('metadata:', df.shape, '| split sizes:', df['split'].value_counts().to_dict())

os.makedirs('data/processed/features', exist_ok=True)
df.to_csv('data/processed/metadata.csv', index=False)

mfcc_arr, mel_arr, valid_lengths = batch_extract(df['filepath'].tolist(), n_jobs=-1, verbose=0)
print('mfcc:', mfcc_arr.shape, '| mel:', mel_arr.shape)

import numpy as np
np.save('data/processed/features/mfcc.npy', mfcc_arr)
np.save('data/processed/features/mel.npy', mel_arr)
np.save('data/processed/features/valid_lengths.npy', valid_lengths)


## 4. SVM baseline

In [ ]:
from src.baseline import get_splits, train_svm
from src.evaluation import evaluate, print_summary, save_result

mfcc = np.load('data/processed/features/mfcc.npy')
X_train, y_train, X_val, y_val, X_test, y_test = get_splits(mfcc, df)
svm_model, scaler, best_params = train_svm(X_train, y_train, X_val, y_val)

test_groups = df[df['split'] == 'test']['dataset'].to_numpy()
test_pred = svm_model.predict(scaler.transform(X_test))
svm_result = evaluate(y_test, test_pred, groups=test_groups)
print_summary('SVM baseline - TEST', svm_result)
save_result({'best_params': best_params, 'test': svm_result}, 'results/svm_baseline.json')


## 5. LSTM-only and CNN-only

In [ ]:
from src.data.parse import LABEL_TO_IDX
from src.data.keras_prep import add_channel_dim, zero_out_padding
from src.models import build_lstm_classifier, build_cnn_classifier
from src.training import train_model, get_predictions

mel = np.load('data/processed/features/mel.npy')
mfcc_masked = zero_out_padding(mfcc, valid_lengths)
mel_ch = add_channel_dim(mel)
y = df['emotion'].map(LABEL_TO_IDX).to_numpy()

train_mask = (df['split'] == 'train').to_numpy()
val_mask = (df['split'] == 'val').to_numpy()
test_mask = (df['split'] == 'test').to_numpy()

lstm_model = build_lstm_classifier()
lstm_model, lstm_history = train_model(
    lstm_model, mfcc_masked[train_mask], y[train_mask], mfcc_masked[val_mask], y[val_mask],
    epochs=60, patience=8, checkpoint_path='models/lstm_only.keras',
)
y_true, y_pred = get_predictions(lstm_model, mfcc_masked[test_mask], y[test_mask])
lstm_result = evaluate(y_true, y_pred, groups=test_groups)
print_summary('LSTM-only - TEST', lstm_result)
save_result({'history': lstm_history, 'test': lstm_result}, 'results/lstm_only.json')


In [ ]:
cnn_model = build_cnn_classifier()
cnn_model, cnn_history = train_model(
    cnn_model, mel_ch[train_mask], y[train_mask], mel_ch[val_mask], y[val_mask],
    epochs=60, patience=8, checkpoint_path='models/cnn_only.keras',
)
y_true, y_pred = get_predictions(cnn_model, mel_ch[test_mask], y[test_mask])
cnn_result = evaluate(y_true, y_pred, groups=test_groups)
print_summary('CNN-only - TEST', cnn_result)
save_result({'history': cnn_history, 'test': cnn_result}, 'results/cnn_only.json')


## 6. CNN-LSTM hybrid (with augmentation)

In [ ]:
from src.features.augment import build_augmented_train_features

train_df = df[train_mask].reset_index(drop=True)
train_labels_orig = train_df['emotion'].map(LABEL_TO_IDX).to_numpy()
_, mel_aug, labels_aug, valid_lengths_aug = build_augmented_train_features(
    train_df['filepath'].tolist(), train_labels_orig, n_jobs=-1
)
mel_train_full = add_channel_dim(np.concatenate([mel[train_mask], mel_aug], axis=0))
labels_train_full = np.concatenate([train_labels_orig, labels_aug], axis=0)
lengths_train_full = np.concatenate([valid_lengths[train_mask], valid_lengths_aug], axis=0).astype(np.int32)
print(f'training set: {len(labels_train_full)} clips ({len(train_df)} original + {len(labels_aug)} augmented)')


In [ ]:
from src.models import build_cnn_lstm_hybrid

x_train = [mel_train_full, lengths_train_full]
x_val = [add_channel_dim(mel[val_mask]), valid_lengths[val_mask].astype(np.int32)]
x_test = [add_channel_dim(mel[test_mask]), valid_lengths[test_mask].astype(np.int32)]

hybrid_model = build_cnn_lstm_hybrid()
hybrid_model, hybrid_history = train_model(
    hybrid_model, x_train, labels_train_full, x_val, y[val_mask],
    epochs=60, patience=8, checkpoint_path='models/cnn_lstm_hybrid.keras',
)
y_true, y_pred = get_predictions(hybrid_model, x_test, y[test_mask])
hybrid_result = evaluate(y_true, y_pred, groups=test_groups)
print_summary('CNN-LSTM hybrid - TEST', hybrid_result)
save_result({'history': hybrid_history, 'test': hybrid_result}, 'results/hybrid.json')


## 7. Compare all four models

In [ ]:
import pandas as pd
import json

rows = []
for name, path in [('SVM', 'results/svm_baseline.json'), ('LSTM-only', 'results/lstm_only.json'),
                    ('CNN-only', 'results/cnn_only.json'), ('CNN-LSTM hybrid', 'results/hybrid.json')]:
    t = json.load(open(path))['test']
    rows.append({
        'model': name, 'test_acc': t['accuracy'],
        'macro_f1': t['classification_report']['macro avg']['f1-score'],
    })
pd.DataFrame(rows).set_index('model').round(4)


## 8. Download the trained checkpoint

Bring the GPU-trained hybrid checkpoint back to your local repo (replacing the CPU-trained one) if
you want the faster-trained weights, or just to have a backup.


In [ ]:
from google.colab import files
files.download('models/cnn_lstm_hybrid.keras')
